In [47]:
import os
import numpy as np
import keras
from keras import layers, Sequential
import tensorflow as tf
from tensorflow import data as tf_data
import matplotlib.pyplot as plt
from keras.layers import Rescaling, RandomFlip, RandomRotation, RandomZoom
import pickle
from pathlib import Path
import statistics
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)


In [10]:
SEED = 42
IMAGE_SIZE = (128, 128)
INPUT_SIZE = (128, 128, 3)
BATCH_SIZE = 16

# SCRIPT_DIR = Path(__file__).resolve().parent
# PROJECT_ROOT = SCRIPT_DIR.parent
PROJECT_ROOT = Path("../")
DATA_DIR = PROJECT_ROOT / "data/processed"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_HISTORY_DIR = MODELS_DIR / "history"

In [3]:
test_ds = keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, 'test'),
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    pad_to_aspect_ratio=True,
    shuffle=False,      # <-- add this
    seed=SEED
)
test_ds = test_ds.map(
    lambda img, label: (img / 255.0, label),
    num_parallel_calls=tf.data.AUTOTUNE
)
test_ds = test_ds.cache()          # optional but good: locks in the order/data
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

Found 564 files belonging to 4 classes.


In [55]:
def getModelHistory(model_name):
    with open(MODELS_HISTORY_DIR / f"{model_name}_history.pkl", "rb") as file:
        loaded_history = pickle.load(file)

    print(f"{model_name} model loaded successfully!")
    return loaded_history

def getModelMetrics(loaded_history):
    training_loss = round(statistics.mean(loaded_history.history['loss']), 4)
    validation_loss = round(statistics.mean(loaded_history.history['val_loss']), 4)
    training_accuracy = round(statistics.mean(loaded_history.history['accuracy']), 4)
    validation_accuracy = round(statistics.mean(loaded_history.history['val_accuracy']), 4)
    training_time = round(sum(loaded_history.history['epoch_time']), 2)

    print("***")
    print(f'training_loss:       {training_loss}')
    print(f'training_accuracy:   {training_accuracy}')
    print(f'validation_loss:     {validation_loss}')
    print(f'validation_accuracy: {validation_accuracy}')
    print(f'total training time: {training_time}s')
    print("***")

def testing(model):
    y_true = np.concatenate([y for x, y in test_ds], axis=0)   # iteration #1 → one shuffle order
    y_pred_probs = model.predict(test_ds)                        # iteration #2 → a DIFFERENT shuffle order

    # For MULTI-CLASS classification (softmax output), uncomment line below:
    y_pred = np.argmax(y_pred_probs, axis=1)

    # --- SCIKIT-LEARN METRICS ---
    # test_loss, test_accuracy = loaded_history.model.evaluate(test_ds, batch_size=32, verbose=1)
    # test_accuracy

    print("=== Performance Metrics ===")
    # 'macro' calculates metrics for each class independently, then takes the average
    # 'weighted' accounts for class imbalance by weighting by the number of true instances
    print(f"Precision: {precision_score(y_true, y_pred, average='macro'):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred, average='macro'):.4f}")
    print(f"F1 Score:  {f1_score(y_true, y_pred, average='macro'):.4f}\n")


    print("=== Confusion Matrix ===")
    print(confusion_matrix(y_true, y_pred))
    print()

    print("=== Classification Report ===")
    print(classification_report(y_true, y_pred))



In [ ]:
# Training Loss
# Validation Loss
# Training Accuracy
# Validation Accuracy
# Training Time
# Test Accuracy

# Precision
# Recall
# F1-score
# Confusion Matrix

In [56]:
models = ['Resnet', "MobileNetV3", "EfficientNetB0", "ConvNeXtTiny"]

for model_name in models:
    model_history = getModelHistory(model_name)
    getModelMetrics(model_history)
    testing(model_history.model)
    print("="*50)
    print("="*50)
    print("="*50)

Resnet model loaded successfully!
***
training_loss:       0.1743
training_accuracy:   0.9517
validation_loss:     0.1528
validation_accuracy: 0.9581
total training time: 184.08s
***
36/36 ━━━━━━━━━━━━━━━━━━━━ 5s 133ms/step
=== Performance Metrics ===
Precision: 0.9610
Recall:    0.9634
F1 Score:  0.9619

=== Confusion Matrix ===
[[139   8   1   2]
 [  2 111   1   0]
 [  0   0 149   1]
 [  2   1   3 144]]

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.97      0.93      0.95       150
           1       0.93      0.97      0.95       114
           2       0.97      0.99      0.98       150
           3       0.98      0.96      0.97       150

    accuracy                           0.96       564
   macro avg       0.96      0.96      0.96       564
weighted avg       0.96      0.96      0.96       564

MobileNetV3 model loaded successfully!
***
training_loss:       1.3761
training_accuracy:   0.2942
validation_loss:     1.368